# Week 4 — EQTransformer output verification
Checks the picks and detections produced across all 60 stations: confirms
per-station counts are in a sane range, checks the total against
expectations, and looks for the 4 September 2019 mainshock as a temporal
sanity check that the pipeline is capturing real seismicity, not noise.

In [ ]:
import os, glob
import pandas as pd

PROJECT_DIR = os.path.expanduser("~/tvz_project")
PICKS_DIR = os.path.join(PROJECT_DIR, "picks", "eqtransformer")

picks_files = sorted(glob.glob(os.path.join(PICKS_DIR, "*_picks.csv")))
detections_files = sorted(glob.glob(os.path.join(PICKS_DIR, "*_detections.csv")))
print(f"{len(picks_files)} picks files, {len(detections_files)} detections files")

In [ ]:
# Exploratory first: confirm the actual column layout before assuming anything
sample = pd.read_csv(picks_files[0])
print("Picks columns:", list(sample.columns))
print(sample.head())

In [ ]:
sample_det = pd.read_csv(detections_files[0])
print("Detections columns:", list(sample_det.columns))
print(sample_det.head())

In [ ]:
# Load everything, using the real column layout confirmed above
picks_df = pd.concat([pd.read_csv(f) for f in picks_files], ignore_index=True)
detections_df = pd.concat([pd.read_csv(f) for f in detections_files], ignore_index=True)

picks_df["station_code"] = picks_df["station"].str.split(".").str[1]
detections_df["station_code"] = detections_df["station"].str.split(".").str[1]

picks_df["time"] = pd.to_datetime(picks_df["time"])
detections_df["start_time"] = pd.to_datetime(detections_df["start_time"])

print(f"Total picks: {len(picks_df)}")
print(f"Total detections: {len(detections_df)}")
print(f"P picks: {(picks_df['phase'] == 'P').sum()}, S picks: {(picks_df['phase'] == 'S').sum()}")
print(f"Probability range: {picks_df['probability'].min():.4f} - {picks_df['probability'].max():.4f}")

In [ ]:
# Per-station summary
summary = picks_df.groupby("station_code").agg(
    n_picks=("time", "count"),
    n_P=("phase", lambda x: (x == "P").sum()),
    n_S=("phase", lambda x: (x == "S").sum()),
    mean_probability=("probability", "mean"),
).reset_index()

det_summary = detections_df.groupby("station_code").agg(n_detections=("start_time", "count")).reset_index()
summary = summary.merge(det_summary, on="station_code", how="outer").fillna(0)
summary = summary.sort_values("n_picks")

pd.set_option("display.max_rows", None)
print(summary.to_string(index=False))

out_csv = os.path.join(PROJECT_DIR, "picks", "eqtransformer_summary.csv")
summary.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
# Sanity checks: flag any stations that look like outliers rather than real variation
print("=== Sanity checks ===")
print(f"Stations with 0 picks: {(summary['n_picks'] == 0).sum()}")
print(f"Mean picks/station: {summary['n_picks'].mean():.0f}, median: {summary['n_picks'].median():.0f}")

low_threshold = summary["n_picks"].quantile(0.05)
print(f"\n5th-percentile pick count: {low_threshold:.0f} — stations at or below it:")
print(summary[summary["n_picks"] <= low_threshold].to_string(index=False))

print(f"\nHighest pick count: {summary['n_picks'].max():.0f} "
      f"({summary.loc[summary['n_picks'].idxmax(), 'station_code']})")

In [ ]:
# Temporal check: does the 4 Sept 2019 mainshock show up as a clear spike?
import datetime

picks_df["date"] = picks_df["time"].dt.date
daily_counts = picks_df.groupby("date").size()

mainshock_date = datetime.date(2019, 9, 4)
window = daily_counts.loc[datetime.date(2019, 8, 30):datetime.date(2019, 9, 10)]
print("Daily pick counts, 30 Aug - 10 Sep 2019:")
print(window)

print(f"\nOverall daily mean across the full window: {daily_counts.mean():.0f}")
print(f"Mainshock day (4 Sep) count: {daily_counts.get(mainshock_date, 0):.0f}")
print(f"Ratio to daily mean: {daily_counts.get(mainshock_date, 0) / daily_counts.mean():.1f}x")